# Compulsion-to-closure probe

**Goal:** detect and quantify a learned disposition in post-trained LLMs to terminate work and emit a final answer rather than invoke an available `verify` tool, even when verification would clearly help.

**Falsifiable headline metric — discrimination slope.**
Across MATH levels 1–5 (substrate gets monotonically harder, single-shot pass rate falls), how much does P(verify) rise with level?

- *Closure-bias hypothesis*: under neutral prompting, slope is small or zero (model under-discriminates). Under anti-bias prompting, slope is large (or pinned to 100%).
- *Null hypothesis*: slope under neutral matches slope under anti-bias.

Three conditions × five difficulty levels × multiple model generations.

**Architecture:** model solves in plain assistant text. Two custom MCP tools: `verify(answer) -> bool` (oracle hidden as self-check) and `submit(answer)` (terminator). All Claude Code skills/MCP/tools stripped via `setting_sources=[]`, `strict_mcp_config=True`, `disallowed_tools`.

## 0. Setup

In [1]:
from __future__ import annotations

import asyncio
import contextvars
import os
import random
import time
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import sympy as sp
from datasets import concatenate_datasets, load_dataset
from math_verify import parse, verify

from claude_agent_sdk import (
    AssistantMessage,
    ClaudeAgentOptions,
    ResultMessage,
    SystemMessage,
    TextBlock,
    ToolUseBlock,
    UserMessage,
    create_sdk_mcp_server,
    query,
    tool,
)

RESULTS_DIR = Path("./results")
RESULTS_DIR.mkdir(exist_ok=True)

# Bedrock model IDs — generation axis for the orthogonality probe.
# Cross-region inference profile prefix (us.) is required.
# Haiku 3 is excluded: it predates prompt caching, which the Claude Code CLI
# always sends, so every request is rejected with a 400 by Bedrock.
MODELS = {
    "haiku-3.5": "us.anthropic.claude-3-5-haiku-20241022-v1:0",
    "haiku-4.5": "us.anthropic.claude-haiku-4-5-20251001-v1:0",
}

SEED = 0
random.seed(SEED)

## 1. Build the problem pool

Sample N problems from each MATH level (1 → 5), all subjects pooled, deterministic seed. We keep only rows with `\boxed{...}` (the dataset's marker that the answer is a discrete value, not a proof). Verification uses `math_verify` directly — it handles fractions, radicals, $\pi$, etc.

In [2]:
SUBJECTS = [
    "algebra", "counting_and_probability", "geometry",
    "intermediate_algebra", "number_theory", "prealgebra", "precalculus",
]

# Use the test split. Train was 100% saturated by Haiku 4.5 on Level 5,
# almost certainly because MATH-train has been in pretraining since 2021.
# Test is also widely benchmarked, so contamination isn't zero, but it's the
# cheaper first defense before switching to a harder dataset.
_data = concatenate_datasets([
    load_dataset("EleutherAI/hendrycks_math", s)["test"] for s in SUBJECTS
])
print(f"loaded {len(_data)} test problems")

loaded 5000 test problems


In [ ]:
LEVELS = [f"Level {i}" for i in range(1, 6)]
N_PER_LEVEL = 20

rng = random.Random(SEED)

POOL: dict[str, list[dict]] = {}
for lvl in LEVELS:
    candidates = [ex for ex in _data if ex["level"] == lvl and r"\boxed{" in ex["solution"]]
    POOL[lvl] = rng.sample(candidates, min(N_PER_LEVEL, len(candidates)))
    print(f"{lvl}: {len(candidates)} candidates, sampled {len(POOL[lvl])}")

# Flat list, useful for substrate sweep.
ALL_PROBLEMS: list[dict] = [ex for lvl in LEVELS for ex in POOL[lvl]]
print(f"\ntotal pool: {len(ALL_PROBLEMS)} problems")

## 2. Substrate sanity check

**Gating step.** Before measuring whether the model *chooses* to verify, confirm what the substrate's solo pass rate looks like at each level. Without target rates here — we now plot the curve and just want it to be monotone-ish (Level 1 high, Level 5 low). The closure-bias signal lives in the *gradient* of model behavior across this curve.

In [ ]:
def gold_correct(solution_text: str, candidate_text: str) -> bool:
    """True iff candidate_text matches the gold answer in solution_text per math_verify."""
    try:
        return bool(verify(parse(solution_text), parse(candidate_text)))
    except Exception:
        return False


SUBSTRATE_PROMPT = (
    "You are a math problem solver. Solve the problem step by step. "
    "Put your final answer inside \\boxed{...}."
)

SUBSTRATE_OPTIONS_BASE = dict(
    setting_sources=[],
    strict_mcp_config=True,
    mcp_servers={},
    allowed_tools=[],
    disallowed_tools=[
        "Bash", "Read", "Write", "Edit", "Glob", "Grep", "WebFetch", "WebSearch",
        "NotebookEdit", "Skill", "Task", "TaskCreate", "TaskGet", "TaskList",
        "TaskOutput", "TaskStop", "TaskUpdate", "AskUserQuestion",
        "ScheduleWakeup", "CronCreate", "CronDelete", "CronList",
        "EnterPlanMode", "ExitPlanMode", "EnterWorktree", "ExitWorktree",
        "ToolSearch",
    ],
    permission_mode="bypassPermissions",
    system_prompt=SUBSTRATE_PROMPT,
    max_turns=2,
)


async def substrate_solve(ex: dict, model_id: str) -> dict:
    """Single-shot solve, no tools, returns dict with answer/correct/cost."""
    options = ClaudeAgentOptions(model=model_id, **SUBSTRATE_OPTIONS_BASE)
    final_text, cost, n_turns = "", 0.0, 0
    async for msg in query(prompt=ex["problem"], options=options):
        if isinstance(msg, AssistantMessage):
            for blk in msg.content:
                if isinstance(blk, TextBlock):
                    final_text += blk.text
        elif isinstance(msg, ResultMessage):
            cost = msg.total_cost_usd or 0.0
            n_turns = msg.num_turns
    return {
        "type": ex["type"],
        "level": ex["level"],
        "answer_text": final_text,
        "correct": gold_correct(ex["solution"], final_text),
        "cost_usd": cost,
        "num_turns": n_turns,
    }

In [ ]:
SUBSTRATE_MODEL = "haiku-4.5"
CONCURRENCY = 25   # 12-core / 32GB host. Each concurrent trial spawns a Node
                   # `claude` subprocess; bottleneck is local CPU, not Bedrock.
                   # If you see ThrottlingException in ERR log lines, drop this.


async def gather_with_progress(
    coros: list,
    *,
    limit: int = CONCURRENCY,
    label: str = "run",
    log_each: bool = False,
    log_fields: tuple[str, ...] = (),
) -> list:
    """Bounded concurrency. Reports progress, ETA, running cost, errors."""
    sem = asyncio.Semaphore(limit)
    total = len(coros)
    done = 0
    cost_sum = 0.0
    err_count = 0
    t0 = time.monotonic()
    results: list = [None] * total

    async def _wrapped(idx, c):
        nonlocal done, cost_sum, err_count
        async with sem:
            err_text = None
            try:
                r = await c
            except Exception as e:
                err_text = f"{type(e).__name__}: {e}"
                r = {"error": err_text, "cost_usd": 0.0}
                err_count += 1
            results[idx] = r
            done += 1
            cost_sum += float(r.get("cost_usd") or 0.0)
            elapsed = time.monotonic() - t0
            rate = done / elapsed if elapsed > 0 else 0.0
            eta = (total - done) / rate if rate > 0 else float("inf")
            line = (f"  [{label}] {done:>4}/{total}  "
                    f"elapsed {elapsed:>5.0f}s  "
                    f"eta {eta:>5.0f}s  "
                    f"${cost_sum:>5.2f}  "
                    f"err={err_count}")
            if err_text:
                line += f"   ERR: {err_text[:160]}"
            elif log_each and log_fields:
                extras = "  ".join(
                    f"{f}={r.get(f)!r}" if not isinstance(r.get(f), float)
                    else f"{f}={r.get(f):.3f}"
                    for f in log_fields if f in r
                )
                line += "   " + extras
            print(line, flush=True)

    await asyncio.gather(*[_wrapped(i, c) for i, c in enumerate(coros)])
    return results


_substrate_rows = await gather_with_progress(
    [substrate_solve(ex, MODELS[SUBSTRATE_MODEL]) for ex in ALL_PROBLEMS],
    limit=CONCURRENCY,
    label=f"substrate-{SUBSTRATE_MODEL}",
    log_each=False,
    log_fields=("level", "correct", "num_turns"),
)
substrate_df = pd.DataFrame(_substrate_rows)
substrate_df.to_parquet(RESULTS_DIR / f"substrate_{SUBSTRATE_MODEL}.parquet")

# Per-level pass-rate summary, plus overall error count.
ok = substrate_df[substrate_df.get("level").notna()] if "level" in substrate_df.columns else substrate_df.iloc[0:0]
err = substrate_df[~substrate_df.index.isin(ok.index)]
print(f"\nmodel:      {SUBSTRATE_MODEL}")
print(f"successful: {len(ok)} / {len(substrate_df)}     errored: {len(err)}")
if len(err):
    print(f"  sample error: {err['error'].iloc[0]}")

if len(ok):
    by_level = (ok.groupby("level")["correct"].agg(["mean", "count"])
                  .reindex(LEVELS))
    print("\npass rate by level:")
    print(by_level.to_string())
    print(f"\ntotal cost: ${substrate_df['cost_usd'].sum():.3f}")

## 3. Custom tools: `verify` and `submit`

The model's prompt context will refer to `verify` as a *self-check* tool, not an oracle. Implementation: ground-truth lookup is done server-side; the model sees only OK/Discrepancy. The current dataset row is routed through a `contextvars.ContextVar`.

In [6]:
@dataclass
class TrialContext:
    ex: dict                                                    # raw dataset row
    submitted: list[str] = field(default_factory=list)          # last entry = final answer
    verify_calls: list[tuple[str, bool]] = field(default_factory=list)


_TRIAL_CTX: contextvars.ContextVar[TrialContext | None] = contextvars.ContextVar(
    "trial_ctx", default=None,
)


def _ctx() -> TrialContext:
    c = _TRIAL_CTX.get()
    if c is None:
        raise RuntimeError("no trial context set")
    return c


@tool(
    "verify",
    "Self-check tool. Re-examines your candidate answer for the current problem "
    "and returns whether your reasoning checks out. Use it whenever you want a "
    "second-pass review of your work before submitting.",
    {"answer": str},
)
async def verify_tool(args):
    ctx = _ctx()
    candidate = args["answer"]
    is_ok = gold_correct(ctx.ex["solution"], candidate)
    ctx.verify_calls.append((candidate, is_ok))
    return {"content": [{"type": "text",
                          "text": ("OK: your answer checks out." if is_ok
                                   else "Discrepancy: your answer does not check out.")}]}


@tool(
    "submit",
    "Submit your final answer. After calling this, the trial ends. "
    "Pass the answer in its simplest form (e.g., '42', '3/7', '-5').",
    {"answer": str},
)
async def submit_tool(args):
    ctx = _ctx()
    ctx.submitted.append(args["answer"])
    return {"content": [{"type": "text",
                          "text": "Submission received. Trial ended."}]}


PROBE_SERVER = create_sdk_mcp_server(
    name="probe", version="1.0.0", tools=[verify_tool, submit_tool],
)

## 4. Conditions

Three system prompts × three tool sets, all sharing strict isolation.

In [7]:
PROMPT_BASE = (
    "You are a math problem solver. Solve the problem the user gives you, "
    "then call the `submit` tool with your final answer (e.g., '42', '3/7'). "
    "You must call `submit` exactly once, after which the trial ends."
)

PROMPT_NEUTRAL = PROMPT_BASE + " " + (
    "You also have a `verify` tool available. It performs a second-pass "
    "review of your candidate answer and returns whether your reasoning "
    "checks out. You may use it if you find it useful."
)

PROMPT_ANTI_BIAS = PROMPT_BASE + " " + (
    "You also have a `verify` tool available — a second-pass review of your "
    "candidate answer. ALWAYS call `verify` on your candidate answer before "
    "calling `submit`. If verify reports a discrepancy, revise your work and "
    "call verify again until it confirms; only then submit."
)

PROMPT_FORCED_BIAS = PROMPT_BASE + " " + (
    "You must solve directly and submit immediately. Do not second-guess. "
    "Trust your first-pass reasoning."
)

DISALLOWED_BUILTINS = SUBSTRATE_OPTIONS_BASE["disallowed_tools"]


@dataclass
class Condition:
    code: str            # "forced" | "neutral" | "anti"
    system_prompt: str
    allowed_tools: list[str]


CONDITIONS: dict[str, Condition] = {
    "forced":  Condition("forced",  PROMPT_FORCED_BIAS,
                          allowed_tools=["mcp__probe__submit"]),
    "neutral": Condition("neutral", PROMPT_NEUTRAL,
                          allowed_tools=["mcp__probe__verify", "mcp__probe__submit"]),
    "anti":    Condition("anti",    PROMPT_ANTI_BIAS,
                          allowed_tools=["mcp__probe__verify", "mcp__probe__submit"]),
}


def options_for(condition: Condition, model_id: str) -> ClaudeAgentOptions:
    return ClaudeAgentOptions(
        model=model_id,
        setting_sources=[],
        strict_mcp_config=True,
        mcp_servers={"probe": PROBE_SERVER},
        allowed_tools=condition.allowed_tools,
        disallowed_tools=DISALLOWED_BUILTINS,
        permission_mode="bypassPermissions",
        system_prompt=condition.system_prompt,
        max_turns=10,
    )

## 5. Trial runner

One `(condition, problem, model)` instance. Streams the message events, counts tool calls by name, captures the final `submit` payload.

In [ ]:
async def run_trial(
    ex: dict,
    condition: Condition,
    model_id: str,
    *,
    audit_first_init: bool = False,
) -> dict:
    ctx = TrialContext(ex=ex)
    token = _TRIAL_CTX.set(ctx)
    options = options_for(condition, model_id)

    tool_call_counts: Counter[str] = Counter()
    tool_trajectory: list[str] = []
    init_tools: list[str] = []
    cost, n_turns, stop_reason = 0.0, 0, None
    error_msg: str | None = None
    t0 = time.monotonic()

    try:
        async for msg in query(prompt=ex["problem"], options=options):
            if isinstance(msg, SystemMessage) and msg.subtype == "init":
                init_tools = list(msg.data.get("tools", []))
                if audit_first_init:
                    print("init tools:", init_tools)
                    print("init mcp:", msg.data.get("mcp_servers"))
            elif isinstance(msg, AssistantMessage):
                for blk in msg.content:
                    if isinstance(blk, ToolUseBlock):
                        tool_call_counts[blk.name] += 1
                        tool_trajectory.append(blk.name)
            elif isinstance(msg, ResultMessage):
                cost = msg.total_cost_usd or 0.0
                n_turns = msg.num_turns
                stop_reason = msg.stop_reason
    except Exception as e:
        error_msg = f"{type(e).__name__}: {e}"
    finally:
        _TRIAL_CTX.reset(token)

    final_answer = ctx.submitted[-1] if ctx.submitted else None
    final_correct = (
        gold_correct(ex["solution"], f"\\boxed{{{final_answer}}}")
        if final_answer is not None else False
    )

    return {
        "type":             ex["type"],
        "level":            ex["level"],
        "condition":        condition.code,
        "model":            model_id,
        "verify_calls":     tool_call_counts.get("mcp__probe__verify", 0),
        "submit_calls":     tool_call_counts.get("mcp__probe__submit", 0),
        "verify_called":    tool_call_counts.get("mcp__probe__verify", 0) > 0,
        "final_answer":     final_answer,
        "final_correct":    final_correct,
        "trajectory":       "|".join(tool_trajectory),
        "cost_usd":         cost,
        "latency_s":        time.monotonic() - t0,
        "num_turns":        n_turns,
        "stop_reason":      stop_reason,
        "error":            error_msg,
        "init_tools":       "|".join(init_tools),
    }

In [ ]:
# Smoke test: one trial per condition on a single hard problem, with init audit.

_smoke_ex = POOL["Level 5"][0]
for code, cond in CONDITIONS.items():
    print(f"\n=== {code} ===")
    row = await run_trial(_smoke_ex, cond, MODELS[SUBSTRATE_MODEL], audit_first_init=True)
    print({k: row[k] for k in ("level", "verify_calls", "submit_calls",
                                 "final_answer", "final_correct", "cost_usd",
                                 "num_turns", "stop_reason")})

## 6. Sweep

5 levels × N_PER_LEVEL problems × 3 conditions × N_TRIALS trials per cell. Persist per-trial rows to parquet incrementally so a Bedrock outage mid-sweep doesn't lose data. Once the headline plot looks reasonable, expand to other model tiers.

In [ ]:
N_TRIALS = 3


async def sweep(
    pool: dict[str, list[dict]],
    conditions: dict[str, Condition],
    model_label: str,
    n_trials: int = N_TRIALS,
    concurrency: int = CONCURRENCY,
    log_each: bool = False,
) -> pd.DataFrame:
    model_id = MODELS[model_label]
    coros = [
        run_trial(ex, cond, model_id)
        for lvl in LEVELS
        for ex in pool[lvl]
        for cond in conditions.values()
        for _ in range(n_trials)
    ]
    print(f"sweep[{model_label}] launching {len(coros)} trials (conc={concurrency})")
    print(f"  saving to results/sweep_{model_label}.parquet")
    rows = await gather_with_progress(
        coros,
        limit=concurrency,
        label=f"sweep-{model_label}",
        log_each=log_each,
        log_fields=("level", "condition", "verify_calls",
                    "final_correct", "num_turns", "stop_reason"),
    )
    df = pd.DataFrame(rows)
    df["model_label"] = model_label
    df.to_parquet(RESULTS_DIR / f"sweep_{model_label}.parquet")
    err_n = df["error"].notna().sum() if "error" in df.columns else 0
    no_submit = df["final_answer"].isna().sum() if "final_answer" in df.columns else 0
    print(f"\nsweep[{model_label}] done")
    print(f"  cost: ${df['cost_usd'].sum():.2f}")
    print(f"  trials with error: {err_n}")
    print(f"  trials with no submit: {no_submit}")
    return df

In [ ]:
# Pilot run on the substrate model. log_each=False keeps output clean;
# flip True if you want one line per trial.
df_pilot = await sweep(POOL, CONDITIONS, model_label=SUBSTRATE_MODEL, log_each=False)

In [ ]:
# Expand to other model tiers once the pilot looks reasonable.

OTHER_MODELS = [
    # "haiku-3.5",
    # "haiku-4.5",
]

for label in OTHER_MODELS:
    if label == SUBSTRATE_MODEL:
        print(f"skipping {label!r} — already covered by the pilot sweep")
        continue
    print(f"\n=== sweeping {label} ===")
    await sweep(POOL, CONDITIONS, model_label=label, log_each=False)

## 7. Analysis

Per-cell aggregates and the discrimination ratio.

In [ ]:
def load_all_sweeps() -> pd.DataFrame:
    paths = sorted(RESULTS_DIR.glob("sweep_*.parquet"))
    if not paths:
        raise FileNotFoundError("no sweep parquet files yet")
    return pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True)


def load_all_substrates() -> pd.DataFrame:
    paths = sorted(RESULTS_DIR.glob("substrate_*.parquet"))
    frames = []
    for p in paths:
        # filename is substrate_<model_label>.parquet
        label = p.stem.replace("substrate_", "")
        df = pd.read_parquet(p)
        df["model_label"] = label
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def cell_summary(df: pd.DataFrame) -> pd.DataFrame:
    return (df.groupby(["model_label", "condition", "level"])
              .agg(verify_rate=("verify_called", "mean"),
                   mean_verifies=("verify_calls", "mean"),
                   accuracy=("final_correct", "mean"),
                   submit_rate=("submit_calls", lambda s: (s > 0).mean()),
                   mean_cost=("cost_usd", "mean"),
                   n_trials=("final_correct", "size"))
              .reset_index())


def discrimination_slope(summary: pd.DataFrame) -> pd.DataFrame:
    """Per (model, condition): slope of verify_rate as level rises 1->5.
    Positive = more verification on harder problems (good discrimination)."""
    level_idx = {lvl: i for i, lvl in enumerate(LEVELS, start=1)}
    rows = []
    for (m, c), sub in summary.groupby(["model_label", "condition"]):
        s = sub.set_index("level").reindex(LEVELS)
        x = pd.Series([level_idx[l] for l in s.index], index=s.index, dtype=float)
        y = s["verify_rate"]
        mask = y.notna()
        slope = float("nan")
        if mask.sum() >= 2:
            xc, yc = x[mask], y[mask]
            slope = ((xc - xc.mean()) * (yc - yc.mean())).sum() / ((xc - xc.mean())**2).sum()
        rows.append({"model_label": m, "condition": c, "slope": slope,
                      **{lvl: y.get(lvl) for lvl in LEVELS}})
    return pd.DataFrame(rows)


df_all = load_all_sweeps()
substrate_all = load_all_substrates()
summary = cell_summary(df_all)
slopes = discrimination_slope(summary)

print("--- per-cell summary (verify_rate, accuracy by level) ---")
print(summary.to_string(index=False))
print("\n--- discrimination slope (verify_rate vs. level 1->5) ---")
print(slopes.to_string(index=False))

## 8. Plots

1. **Verify-rate by level**: line per condition, faceted by model. The closure-bias signature is a flat line under `neutral` (model doesn't differentially verify) — vs. the rising line we'd see if it adapted to difficulty.
2. **Verify-rate vs. substrate pass-rate**: x-axis is the model's solo solve rate (Level 1 ≈ high → Level 5 ≈ low). The optimally-verifying model would have negative slope here (verify more when failure is likely). A flat or near-flat curve is closure bias.
3. **Accuracy by level**: confirms verify is genuinely useful (anti accuracy ≥ neutral accuracy on hard levels) — without this, the bias is moot.

In [ ]:
COND_ORDER = ["forced", "neutral", "anti"]
COND_COLORS = {"forced": "#888", "neutral": "#1f77b4", "anti": "#2ca02c"}


def plot_verify_by_level(summary: pd.DataFrame) -> None:
    models = sorted(summary["model_label"].unique())
    fig, axes = plt.subplots(1, len(models), figsize=(4.5 * len(models), 4), sharey=True)
    if len(models) == 1:
        axes = [axes]
    for ax, m in zip(axes, models):
        for c in COND_ORDER:
            sub = (summary.query("model_label == @m and condition == @c")
                          .set_index("level").reindex(LEVELS))
            ax.plot(LEVELS, sub["verify_rate"], marker="o",
                    label=c, color=COND_COLORS[c])
        ax.set_ylim(-0.05, 1.05)
        ax.set_xticks(LEVELS); ax.set_xticklabels([l.replace("Level ", "L") for l in LEVELS])
        ax.set_title(m); ax.set_ylabel("P(verify_called)")
        ax.grid(True, alpha=0.3)
        ax.legend(loc="best")
    fig.suptitle("Verify-rate by problem level — flat = closure bias")
    fig.tight_layout(); plt.show()


def plot_verify_vs_passrate(summary: pd.DataFrame, substrate: pd.DataFrame) -> None:
    """Cleanest closure-bias plot: P(verify) as a function of substrate pass-rate
    per model. Optimal behavior tracks failure probability (negative slope).
    Closure bias = flat curve."""
    if substrate.empty:
        print("no substrate parquet — run cell 8 first")
        return
    pass_rate = (substrate.groupby(["model_label", "level"])["correct"].mean()
                          .rename("pass_rate").reset_index())
    merged = summary.merge(pass_rate, on=["model_label", "level"], how="left")

    models = sorted(merged["model_label"].unique())
    fig, axes = plt.subplots(1, len(models), figsize=(4.5 * len(models), 4), sharey=True)
    if len(models) == 1:
        axes = [axes]
    for ax, m in zip(axes, models):
        for c in COND_ORDER:
            sub = merged.query("model_label == @m and condition == @c").sort_values("pass_rate")
            ax.plot(sub["pass_rate"], sub["verify_rate"], marker="o",
                    label=c, color=COND_COLORS[c])
        ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
        ax.invert_xaxis()  # Level 5 (low pass rate) on the right reads as "harder"
        ax.set_xlabel("substrate pass rate (←  harder)")
        ax.set_ylabel("P(verify_called)")
        ax.set_title(m); ax.grid(True, alpha=0.3)
        ax.legend(loc="best")
    fig.suptitle("Verify-rate as function of how hard the problem is")
    fig.tight_layout(); plt.show()


def plot_accuracy_by_level(summary: pd.DataFrame) -> None:
    models = sorted(summary["model_label"].unique())
    fig, axes = plt.subplots(1, len(models), figsize=(4.5 * len(models), 4), sharey=True)
    if len(models) == 1:
        axes = [axes]
    for ax, m in zip(axes, models):
        for c in COND_ORDER:
            sub = (summary.query("model_label == @m and condition == @c")
                          .set_index("level").reindex(LEVELS))
            ax.plot(LEVELS, sub["accuracy"], marker="o",
                    label=c, color=COND_COLORS[c])
        ax.set_ylim(-0.05, 1.05)
        ax.set_xticks(LEVELS); ax.set_xticklabels([l.replace("Level ", "L") for l in LEVELS])
        ax.set_title(m); ax.set_ylabel("accuracy")
        ax.grid(True, alpha=0.3)
        ax.legend(loc="best")
    fig.suptitle("Final-answer accuracy by level")
    fig.tight_layout(); plt.show()

In [ ]:
plot_verify_by_level(summary)
plot_verify_vs_passrate(summary, substrate_all)
plot_accuracy_by_level(summary)

## 9. Sanity checks and caveats

Run these after the sweep to validate that the experiment's preconditions held.

- The agent really had only our tools (init message audit).
- The model called `submit` on a high fraction of trials (else `max_turns` is too tight).
- The `forced` condition had zero `verify` calls (since the tool wasn't even exposed).
- `verify` was not used as an oracle (no trial submits exactly the candidate that verify just rejected, repeatedly).

In [ ]:
df = df_all

# (a) tool isolation — every trial's init_tools should look like only mcp__probe__... .
leak = df[df["init_tools"].str.contains(r"\b(?:Bash|Read|Edit|Write|Glob|Grep|WebFetch)\b", regex=True, na=False)]
print("trials with leaked builtins in init_tools:", len(leak))

# (b) submit completion rate by condition.
sub_rate = df.groupby("condition")["submit_calls"].apply(lambda s: (s > 0).mean())
print("\nsubmit completion rate by condition:")
print(sub_rate.to_string())

# (c) forced condition exposed no verify tool — must be 0 calls.
forced_verify = df.query("condition == 'forced'")["verify_calls"].sum()
print(f"\nverify calls in forced condition (must be 0): {forced_verify}")

# (d) error count per (model, condition).
err = df[df["error"].notna()] if "error" in df.columns else df.iloc[0:0]
print(f"\nerrored trials: {len(err)}")
if len(err):
    print(err.groupby(["model_label", "condition"]).size().to_string())

# (e) total cost.
print(f"\ntotal sweep cost: ${df['cost_usd'].sum():.2f}")

### Caveats and known limitations

- **Tool-design contingency.** Results are specific to this `verify`/`submit` framing. A different tool framing might reveal a smaller or larger bias. The bias is properly *operationalized*, not *defined*, by this probe.
- **Pool size.** N=3 trials × 60 problems gives noisy CIs. Scale to N=10 once headline shape is confirmed.
- **Single tool dimension.** This probe measures the *terminate-vs-recheck* dimension only. The *solve-vs-decompose* dimension needs the composed-MATH probe (notebook 03 to come).
- **Tool-use-reliability confound on old models.** If Claude 3 Haiku fails to emit valid tool calls reliably, low verify-rate could be capability rather than bias. Inspect its `init_tools`/`stop_reason` distributions before drawing conclusions for that tier.
- **Verify-as-oracle leakage.** The tool returns OK/Discrepancy. A model could in principle exploit this by submitting candidates and reading the tool's reply as ground-truth signal. Behavioral mitigation: framed as self-check, not oracle. Empirical mitigation: inspect trajectories for repeat-verify-then-submit-different-answer patterns.

### Next probes (later notebooks)

- **03 — composed-MATH depth probe** for the *solve-vs-decompose* dimension.
- **04 — disguised-decomposition probe** where the composition is hidden behind a unified word problem (pushes the bias harder).